## Download JPEGs and Calculate Extents

Prepare all photos in a selected subset for georeferencing.

- Download original JPEGs
- Read image dimensions and DPI
- Calculate approximate horizontal ground extents
- Save preparation results to a separate tracker GeoPackage

Existing valid JPEGs are reused rather than downloaded again.

In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime, timezone
from tqdm import tqdm
import shutil

from single_photo_workflow import (
    download_photo,
    get_image_metadata,
    calculate_ground_extent,
    update_tracker,
    ground_to_mercator_extent
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

### Load subset table and make paths

In [14]:
SUBSET_PATH = Path(
    "data/historic_airphoto_index/subsets/"
    "subset_20000_carver_37_38_40_51_57_63.gpkg"
)

LAYER = "photos"

# Create a separate GeoPackage for tracking processing progress
TRACKER_PATH = SUBSET_PATH.with_name(
    f"{SUBSET_PATH.stem}_tracker.gpkg"
)

# define subfolder for downloading images, using the geopackage name
DOWNLOAD_DIR = Path("data/downloads") / SUBSET_PATH.stem

### Make tracker geopackage and download directory

In [4]:
# Create tracker gpkg, ONLY if it doesn't already exist
# This is a copy of the subset table, which is preserved blank for input. 

if not TRACKER_PATH.exists():
    shutil.copy2(SUBSET_PATH, TRACKER_PATH)
    print(f"Created tracker: {TRACKER_PATH.name}")
else:
    print(f"Using existing tracker: {TRACKER_PATH.name}")

Created tracker: subset_20000_carver_37_38_40_51_57_63_tracker.gpkg


In [5]:
# Make download directory if it doesn't already exist
# exist_ok makes it succeed silently and not do anything if it already exists
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

### Load tracker, select unfinished photos

In [6]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

In [7]:
print(tracker_gdf.crs)
print(tracker_gdf.geometry.iloc[0])
print(tracker_gdf[["latitude"]].head())

EPSG:3857
POINT (-10465155.1256 5579058.785099998)
    latitude
0  44.729640
1  44.741189
2  44.752738
3  44.764287
4  44.775835


In [8]:
tracker_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,dpi_x,dpi_y,width_px,height_px,worldfile_json,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1937_wm-26-2368,None,Carver,1937,wm-26-2368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,None,None,None,None,None,44.729640,POINT (-10465155.126 5579058.785)
1,carver_1937_wm-26-2369,None,Carver,1937,wm-26-2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,None,None,None,None,None,44.741189,POINT (-10465099.911 5580868.601)
2,carver_1937_wm-26-2370,None,Carver,1937,wm-26-2370,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,None,None,None,None,None,44.752738,POINT (-10465044.585 5582678.778)
3,carver_1937_wm-26-2371,None,Carver,1937,wm-26-2371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,None,None,None,None,None,44.764287,POINT (-10464989.371 5584489.317)
4,carver_1937_wm-26-2372,None,Carver,1937,wm-26-2372,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,None,None,None,None,None,44.775835,POINT (-10464934.045 5586300.061)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,carver_1963_wm-1dd-111,None,Carver,1963,wm-1dd-111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-111.jpg,Carver,92355,None,None,None,None,None,44.702124,POINT (-10428717.028 5574748.276)
867,carver_1963_wm-1dd-113,None,Carver,1963,wm-1dd-113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-113.jpg,Carver,92356,None,None,None,None,None,44.702615,POINT (-10424133.337 5574825.175)
868,carver_1963_wm-1dd-228,None,Carver,1963,wm-1dd-228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-228.jpg,Carver,92362,None,None,None,None,None,44.673257,POINT (-10435676.055 5570228.328)
869,carver_1963_wm-1dd-230,None,Carver,1963,wm-1dd-230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-230.jpg,Carver,92363,None,None,None,None,None,44.673322,POINT (-10431122.308 5570238.503)


In [9]:
# Set to True to re-prepare all photos, regardless of status.
# does not re-download the photo if it exists. to do that, delete the files
REPREPARE_ALL = False

if REPREPARE_ALL:
    photos_gdf = tracker_gdf.copy()
else:
    photos_gdf = tracker_gdf.loc[
        ~tracker_gdf["status"].isin(["prepared", "georeferenced"])
    ].copy()

print(f"Total photos in subset: {len(tracker_gdf)}")
print(f"Photos to prepare: {len(photos_gdf)}")
print(f"Download directory: {DOWNLOAD_DIR}")

display(photos_gdf.head())

Total photos in subset: 871
Photos to prepare: 871
Download directory: data\downloads\subset_20000_carver_37_38_40_51_57_63


,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,dpi_x,dpi_y,width_px,height_px,worldfile_json,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1937_wm-26-2368,None,Carver,1937,wm-26-2368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,None,None,None,None,None,44.729640,POINT (-10465155.126 5579058.785)
1,carver_1937_wm-26-2369,None,Carver,1937,wm-26-2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,None,None,None,None,None,44.741189,POINT (-10465099.911 5580868.601)
2,carver_1937_wm-26-2370,None,Carver,1937,wm-26-2370,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,None,None,None,None,None,44.752738,POINT (-10465044.585 5582678.778)
3,carver_1937_wm-26-2371,None,Carver,1937,wm-26-2371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,None,None,None,None,None,44.764287,POINT (-10464989.371 5584489.317)
4,carver_1937_wm-26-2372,None,Carver,1937,wm-26-2372,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,None,None,None,None,None,44.775835,POINT (-10464934.045 5586300.061)


### Download JPEG, read metadata, calculate extent

In [23]:
# create an empty list to collect results for each photo. 
results = []

# loop through photos_gdf, processing one photo (row) at a time. 
for _, photo in tqdm(
    photos_gdf.iterrows(),
    total=len(photos_gdf),
    desc="Preparing photos"
):
    # get the unique but friendly photo identifier
    photo_id = photo["cty_year_id"]

    # and use it to construct the download path
    image_path = DOWNLOAD_DIR / f"{photo_id}_original.jpg"

    # Create a dictionary to store this photo's processing results.
    # Default values are replaced as each step succeeds.
    result = {
        "source_id": photo["source_id"],
        "width_px": None,
        "height_px": None,
        "dpi_x": None,
        "dpi_y": None,
        "ground_width": None,
        "ground_height": None,
        "wm_width": None,
        "wm_height": None,
        "date_prepared": None,
        "error": None,
    }
    
    try:
        # download JPEG or reuse if already downloaded
        image_path = download_photo(
            url=photo["url"],
            output_path=image_path,
        )

        # record download success and date

        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date()

        # read metadata from the downloaded image
        metadata = get_image_metadata(image_path)

        # update that dictionary
        result.update(metadata)

         # Calculate horizontal ground coverage
        result["ground_width"] = calculate_ground_extent(
            pixels=metadata["width_px"],
            dpi=metadata["dpi_x"],
            scale=photo["scale"],
        )

        # Calculate vertical ground coverage.
        result["ground_height"] = calculate_ground_extent(
            pixels=metadata["height_px"],
            dpi=metadata["dpi_y"],
            scale=photo["scale"],
        )

         # Convert both dimensions to Web Mercator.
        result["wm_width"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude=photo["latitude"],
        )

        result["wm_height"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_height"],
            latitude=photo["latitude"],
        )

    except Exception as e:
        # If any step fails, record the error and continue
        result["error"] = str(e)

    # Save this photo's result dict to the list of dicts
    results.append(result)

print("CELL FINISHED", flush=True)

Preparing photos: 100%|██████████| 871/871 [00:00<00:00, 1893.68it/s]

CELL FINISHED


In [24]:
# convert list of results dicts into a dataframe
results_df = pd.DataFrame(results)

In [27]:
results_df = pd.DataFrame(results)

print(results_df.dtypes)

print(results_df[
    ["ground_width", "ground_height", "wm_width", "wm_height"]
].head())

source_id          int64
width_px           int64
height_px          int64
dpi_x            float64
dpi_y            float64
ground_width      object
ground_height     object
wm_width          object
wm_height         object
date_prepared     object
error                str
dtype: object
  ground_width ground_height wm_width wm_height
0         None          None     None      None
1         None          None     None      None
2         None          None     None      None
3         None          None     None      None
4         None          None     None      None


In [25]:
# examine
pd.set_option('display.max_rows', 40)

results_df

,source_id,width_px,height_px,dpi_x,dpi_y,ground_width,ground_height,wm_width,wm_height,date_prepared,error
0,91674,7105,5671,800.0,800.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
1,91675,7107,5676,800.0,800.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
2,91676,7174,5671,800.0,800.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
3,91677,7110,5679,800.0,800.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
4,91678,7038,5711,800.0,800.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
...,...,...,...,...,...,...,...,...,...,...,...
866,92355,5453,5475,600.0,600.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
867,92356,5477,5475,600.0,600.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
868,92362,5487,5451,600.0,600.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."
869,92363,5479,5443,600.0,600.0,None,None,None,None,2026-11-04,"Width, DPI, and scale must be positive numbers."


In [26]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

TypeError: Invalid value '[None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None None None None None
 None None None None None None None None None None 

In [25]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,dpi_x,dpi_y,width_px,height_px,worldfile_json,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1937_wm-26-2368,prepared,Carver,1937,wm-26-2368,4511.674999999999,3601.085,6350.57649697092,5068.841564295861,20000,800.0,800.0,7105,5671,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,2026-11-04,None,None,None,NaN,44.729640,POINT (-10465155.126 5579058.785)
1,carver_1937_wm-26-2369,prepared,Carver,1937,wm-26-2369,4512.945,3604.2599999999998,6353.632921854273,5074.3239713585,20000,800.0,800.0,7107,5676,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,2026-11-04,None,None,None,NaN,44.741189,POINT (-10465099.911 5580868.601)
2,carver_1937_wm-26-2370,prepared,Carver,1937,wm-26-2370,4555.489999999999,3601.085,6414.812208153059,5070.867024315027,20000,800.0,800.0,7174,5671,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,2026-11-04,None,None,None,NaN,44.752738,POINT (-10465044.585 5582678.778)
3,carver_1937_wm-26-2371,prepared,Carver,1937,wm-26-2371,4514.849999999999,3606.165,6358.855857537255,5079.035501400011,20000,800.0,800.0,7110,5679,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,2026-11-04,None,None,None,NaN,44.764287,POINT (-10464989.371 5584489.317)
4,carver_1937_wm-26-2372,prepared,Carver,1937,wm-26-2372,4469.129999999999,3626.4849999999997,6295.721017087898,5108.67614785294,20000,800.0,800.0,7038,5711,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,2026-11-04,None,None,None,NaN,44.775835,POINT (-10464934.045 5586300.061)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,carver_1963_wm-1dd-111,prepared,Carver,1963,wm-1dd-111,4616.873333333333,4635.5,6495.56292673825,6521.76912229817,20000,600.0,600.0,5453,5475,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-111.jpg,Carver,92355,2026-11-04,None,None,None,NaN,44.702124,POINT (-10428717.028 5574748.276)
867,carver_1963_wm-1dd-113,prepared,Carver,1963,wm-1dd-113,4637.193333333334,4635.5,6524.2068352573315,6521.8244336377375,20000,600.0,600.0,5477,5475,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-113.jpg,Carver,92356,2026-11-04,None,None,None,NaN,44.702615,POINT (-10424133.337 5574825.175)
868,carver_1963_wm-1dd-228,prepared,Carver,1963,wm-1dd-228,4645.66,4615.18,6532.80690046775,6489.945400847404,20000,600.0,600.0,5487,5451,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-228.jpg,Carver,92362,2026-11-04,None,None,None,NaN,44.673257,POINT (-10435676.055 5570228.328)
869,carver_1963_wm-1dd-230,prepared,Carver,1963,wm-1dd-230,4638.886666666666,4608.406666666667,6523.28943931267,6480.427891618701,20000,600.0,600.0,5479,5443,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-230.jpg,Carver,92363,2026-11-04,None,None,None,NaN,44.673322,POINT (-10431122.308 5570238.503)
